In [23]:
!pip install -q transformers datasets torchaudio librosa scikit-learn accelerate torchcodec

In [24]:
import os
import random
import torch
import torchaudio
import numpy as np


from torch.utils.data import Dataset
from transformers import (
Wav2Vec2Processor,
Wav2Vec2ForSequenceClassification,
TrainingArguments,
DataCollatorWithPadding,
Trainer
)


from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [25]:
# -------------------------
# Configuración general
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_PATH = "/content/dataset"
SAMPLE_RATE = 16000
NUM_LABELS = 3
SEED = 42


random.seed(SEED)
torch.manual_seed(SEED)

In [26]:
# -------------------------
# Mapeo de emociones (3 clases)
# -------------------------
emotion2id = {
    "happiness": 0,      # Positivo
    "neutral": 1,      # Neutral
    "sadness": 2,     # Negativo
    "anger": 2,
    "disgust": 2,
    "fear": 2
}

id2emotion = {
    0: "positivo",
    1: "neutral",
    2: "negativo"
}

In [27]:
# -------------------------
# Dataset
# -------------------------
class EmotionDataset(Dataset):
    def __init__(self, files, data_dir, processor):
        self.files = files
        self.data_dir = data_dir
        self.processor = processor

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filename = self.files[idx]
        path = os.path.join(self.data_dir, filename)

        # Etiqueta desde el nombre
        name = os.path.splitext(filename)[0]   # elimina extensión
        emotion = name.split("_")[0].lower()
        label = emotion2id[emotion]

        # Cargar audio
        speech, sr = torchaudio.load(path)

        if sr != SAMPLE_RATE:
            speech = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(speech)

        speech = speech.mean(dim=0)  # mono

        inputs = self.processor(
            speech.numpy(),
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=False
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }


In [28]:
# -------------------------
# Modelo y procesador
# -------------------------
MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-spanish"

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

#data_collator = data_collator

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to(DEVICE)

# Congelar extractor (clave para corpus pequeño)
for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-spanish
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 
projector.bias    | MISSING    | 
projector.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [29]:
# -------------------------
# División del corpus
# -------------------------
all_files = [f for f in os.listdir(DATASET_PATH) if f.lower().endswith(".wav")]
random.shuffle(all_files)

print("Archivos detectados:", len(all_files))
print(all_files[:5])


train_size = int(0.8 * len(all_files))
train_files = all_files[:train_size]
test_files = all_files[train_size:]

print(f"Train: {len(train_files)} | Test: {len(test_files)}")

train_dataset = EmotionDataset(train_files, DATASET_PATH, processor)
test_dataset = EmotionDataset(test_files, DATASET_PATH, processor)


Archivos detectados: 862
['Fear_M_B_ataque.wav', 'Disgust_M_A_basta_ya.wav', 'Neutral_C_A_lento.wav', 'Anger_M_B_ira.wav', 'Fear_M_A_ayer.wav']
Train: 689 | Test: 173


In [30]:
# -------------------------
# Métricas
# -------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1-score": f1
    }

In [31]:
# -------------------------
# Argumentos de entrenamiento
# -------------------------
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=8,
    warmup_steps=10,
    logging_dir="./logs",
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1-score",
    fp16=False
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [32]:
# -------------------------
# Trainer
# -------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=processor,
    compute_metrics=compute_metrics
)
# -------------------------
# Entrenamiento y evaluación
# -------------------------
trainer.train()

results = trainer.evaluate()
print("\nResultados finales:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1-score
1,2.389153,1.235732,0.653179,0.553722,0.653179,0.550328
2,1.658931,1.118354,0.641618,0.775592,0.641618,0.662004
3,0.017379,1.299077,0.722543,0.878674,0.722543,0.743937
4,1.230642,0.650206,0.878613,0.886835,0.878613,0.875613
5,0.003048,0.644347,0.890173,0.918788,0.890173,0.895599
6,0.003257,0.409904,0.930636,0.940704,0.930636,0.932048
7,1.415755,0.443569,0.919075,0.924213,0.919075,0.918786
8,0.002552,0.397568,0.924855,0.934580,0.924855,0.926528


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Resultados finales:
eval_loss: 0.4099
eval_accuracy: 0.9306
eval_precision: 0.9407
eval_recall: 0.9306
eval_f1-score: 0.9320
eval_runtime: 4.0264
eval_samples_per_second: 42.9670
eval_steps_per_second: 42.9670
epoch: 8.0000
